In [ ]:
!pip install -q pandas scikit-learn
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
#Settings

TRAIN_FILE = "Additional.csv"     # training only
DEV_FILE = "Exploration.csv"      # model selection / debugging only
TEST_FILE = "Joint.csv"           # final evaluation only

TEXT_COL = "Text"
LABEL_COL = "Label"
POSITIVE_LABEL = "APPROPRIATE"
MAKE_BINARY = True                 # set to False for original labels
FINAL_TRAIN_ON_DEV = False         # keep False for the cleanest methodology

In [ ]:
#LOAD DATA
def load_data(path):
    df = pd.read_csv(path)
    df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
    df[TEXT_COL] = df[TEXT_COL].astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(str)
    return df

train_df = load_data(TRAIN_FILE)
dev_df = load_data(DEV_FILE)
test_df = load_data(TEST_FILE)

print("\nTrain:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))

In [ ]:
#Binary label mapping

POSITIVE_LABEL = "APPROPRIATE"
NEGATIVE_LABEL = "NOT_APPROPRIATE"

def make_binary(df):
    df = df.copy()
    df[LABEL_COL] = df[LABEL_COL].apply(
        lambda x: POSITIVE_LABEL if x == POSITIVE_LABEL else NEGATIVE_LABEL
    )
    return df

train_df = make_binary(train_df)
dev_df = make_binary(dev_df)
test_df = make_binary(test_df)

In [ ]:
#SPLITS
X_train = train_df[TEXT_COL]
y_train = train_df[LABEL_COL]

X_dev = dev_df[TEXT_COL]
y_dev = dev_df[LABEL_COL]

X_test = test_df[TEXT_COL]
y_test = test_df[LABEL_COL]

print("Dataset sizes")
print(f"  Train: {len(train_df)}")
print(f"  Dev:   {len(dev_df)}")
print(f"  Test:  {len(test_df)}")
print("\nTrain label distribution")
print(y_train.value_counts())
print("\nDev label distribution")
print(y_dev.value_counts())
print("\nTest label distribution")
print(y_test.value_counts())

In [ ]:
#EVALUATION HELPERS
def get_scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def print_report(name, y_true, y_pred):
    scores = get_scores(y_true, y_pred)
    print(f"\n=== {name} ===")
    print(f"Accuracy: {scores['accuracy']:.3f}")
    print(f"Macro F1: {scores['macro_f1']:.3f}")
    print(classification_report(y_true, y_pred, zero_division=0))

def evaluate_on_split(model_name, model, X_split, y_split, split_name):
    preds = model.predict(X_split)
    print_report(f"{model_name} | {split_name}", y_split, preds)
    return preds, get_scores(y_split, preds)

In [ ]:
#Models: Baseline, LinearSVC, Naive
def build_baseline():
    return DummyClassifier(strategy="most_frequent")

def build_svm(stop_words="english", ngram_range=(1, 1), min_df=1, C=0.1):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words=stop_words,
            ngram_range=ngram_range,
            min_df=min_df
        )),
        ("clf", LinearSVC(
            C=C,
            class_weight="balanced",
            random_state=42
        )),
    ])

def build_nb(stop_words="english", ngram_range=(1, 1), min_df=1, alpha=0.2):
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True,
            stop_words=stop_words,
            ngram_range=ngram_range,
            min_df=min_df
        )),
        ("clf", ComplementNB(alpha=alpha)),
    ])
models = {
    "Majority baseline": build_baseline(),
    "Linear SVM": build_svm(),
    "Complement Naive Bayes": build_nb(),
}


In [ ]:
#DEV PHASE (MODEL SELECTION)
dev_results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    _, scores = evaluate_on_split(name, model, X_dev, y_dev, "DEV")
    dev_results.append((name, scores["accuracy"], scores["macro_f1"]))

print("\nDEV SUMMARY ")
print(f"{'Model':<28} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 52)
for name, acc, macro_f1 in dev_results:
    print(f"{name:<28} {acc:>10.3f} {macro_f1:>10.3f}")

best_model_name = sorted(dev_results, key=lambda x: x[2], reverse=True)[0][0]
best_model = trained_models[best_model_name]
print(f"\nBest model on DEV (by Macro F1): {best_model_name}")

In [ ]:
test_pred = best_model.predict(X_test)

ConfusionMatrixDisplay.from_predictions(y_test, test_pred)

WHat the conufion matrix shows us:
The model misses many inappropriate responses.
This is typical when the dataset is imbalanced
and minority class has few examples

In [ ]:
#Macro-F1 Comparison Bar Chart
import matplotlib.pyplot as plt

models = ["Baseline", "Naive Bayes", "SVM"]
scores = [0.463, 0.612, 0.708]

plt.bar(models, scores)
plt.ylabel("Macro F1")
plt.title("Model Comparison")
plt.show()

In [ ]:
#FINAL TEST PHASE
#IMPORTANT: Test is used only once, after model selection is finished.

if FINAL_TRAIN_ON_DEV and best_model_name != "Majority baseline":
    if best_model_name == "Linear SVM":
        best_model = build_svm()
    elif best_model_name == "Complement Naive Bayes":
        best_model = build_nb()
    best_model.fit(pd.concat([X_train, X_dev]), pd.concat([y_train, y_dev]))
    print("\nRetrained best model on TRAIN + DEV before final TEST evaluation.")
else:
    print("\nUsing the model trained on TRAIN only for final TEST evaluation.")

# Evaluate all models on test for comparison, but DO NOT use these scores for new tuning.
test_results = []
all_test_predictions = {}
for name, model in trained_models.items():
    preds, scores = evaluate_on_split(name, model, X_test, y_test, "TEST")
    all_test_predictions[name] = preds
    test_results.append((name, scores["accuracy"], scores["macro_f1"]))

print("\n=== TEST SUMMARY ===")
print(f"{'Model':<28} {'Accuracy':>10} {'Macro F1':>10}")
print("-" * 52)
for name, acc, macro_f1 in test_results:
    print(f"{name:<28} {acc:>10.3f} {macro_f1:>10.3f}")


In [ ]:
#ERROR ANALYSIS
def error_analysis(texts, gold, pred, model_name, n=10):
    error_df = pd.DataFrame({
        "Text": list(texts),
        "Gold": list(gold),
        "Predicted": list(pred),
    })
    errors = error_df[error_df["Gold"] != error_df["Predicted"]].copy()
    print(f"\n=== ERROR ANALYSIS: {model_name} ===")
    if len(errors) == 0:
        print("No errors on this split.")
        return errors
    print(errors.head(n).to_string(index=False))
    return errors

best_test_pred = all_test_predictions[best_model_name]
error_df = error_analysis(X_test, y_test, best_test_pred, best_model_name, n=10)

print("\n=== CONFUSION MATRIX ===")
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, best_test_pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"gold_{l}" for l in labels], columns=[f"pred_{l}" for l in labels])
print(cm_df)